# Generative Adversarial Networks (GANs) — Runnable Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeevchiran/learnings-ai-ml/blob/main/notebook/gan/gan-lab.ipynb)

Companion notebook for the **Generative Adversarial Networks** track (`gan-m1` … `gan-m11`). Builds a DCGAN from scratch in PyTorch, trains it on Fashion-MNIST, and re-derives the numeric claims made across the track.

Runs on **CPU in a few minutes** or on a Colab T4 GPU in under a minute per epoch.

| Part | Topic | What runs |
|---|---|---|
| 1 | `gan-m2` | The optimal discriminator D*(x) = p_data/(p_data+p_g), verified on a 1D toy Gaussian problem |
| 2 | `gan-m3` | Vanishing vs. non-saturating gradient magnitude, computed directly |
| 3 | `gan-m5`, `gan-m10` | DCGAN Generator and Discriminator, built from scratch |
| 4 | `gan-m10` | The full non-saturating training loop on Fashion-MNIST |
| 5 | `gan-m10` | Generated samples after training |
| 6 | `gan-m6` | WGAN-GP gradient penalty, verified against a toy critic |
| 7 | `gan-m8` | CycleGAN's cycle-consistency loss, verified as trivially zero for the identity mapping |
| 8 | `gan-m9` | Toy Inception-Score / FID proxies, showing IS understating a collapsed generator |
| 9 | `gan-m11` | Latent-space interpolation on the trained generator |

## 0. Setup & Dependencies

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Active device:   {DEVICE}')

---
# Part 1 — The Optimal Discriminator on a 1D Toy Problem (`gan-m2`)

For a fixed real distribution and a fixed generator distribution, the optimal discriminator has a closed form:

$$D^*(x) = \frac{p_{data}(x)}{p_{data}(x) + p_g(x)}$$

We verify that as $p_g \to p_{data}$, $D^*(x) \to 1/2$ everywhere — the Module 2 claim.

In [ ]:
def gaussian_pdf(x, mean, std):
    return torch.exp(-0.5 * ((x - mean) / std) ** 2) / (std * np.sqrt(2 * np.pi))

x = torch.linspace(-6, 6, 400)

def optimal_discriminator(x, g_mean, g_std, real_mean=0.0, real_std=1.0):
    p_real = gaussian_pdf(x, real_mean, real_std)
    p_fake = gaussian_pdf(x, g_mean, g_std)
    return p_real / (p_real + p_fake)

d_bad_g = optimal_discriminator(x, g_mean=-3.2, g_std=0.4)   # untrained generator, far from real data
d_good_g = optimal_discriminator(x, g_mean=0.0, g_std=1.0)   # generator matches p_data exactly

print(f'Max |D*(x) - 0.5| with a bad generator:  {(d_bad_g - 0.5).abs().max().item():.4f}')
print(f'Max |D*(x) - 0.5| with a matched generator: {(d_good_g - 0.5).abs().max().item():.6f}')
assert (d_good_g - 0.5).abs().max().item() < 1e-6, 'D*(x) must equal 0.5 everywhere once p_g = p_data'
print('✓ Nash equilibrium confirmed: D*(x) collapses to 0.5 once the generator matches the real distribution.')

plt.figure(figsize=(8, 4))
plt.plot(x, d_bad_g, label='D*(x), untrained G')
plt.plot(x, d_good_g, label='D*(x), G matched to p_data')
plt.axhline(0.5, color='gray', linestyle='--', linewidth=1)
plt.title('Optimal Discriminator Before vs. After Convergence')
plt.xlabel('x'); plt.ylabel('D*(x)'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

---
# Part 2 — Vanishing vs. Non-Saturating Gradient (`gan-m3`)

We compare $\frac{d}{dy}\log(1-y)$ (original) against $\frac{d}{dy}\log(y)$ (non-saturating) at $y = D(G(z))$ close to 0 — the regime an untrained generator starts in.

In [ ]:
y = torch.tensor(0.02, requires_grad=True)  # D(G(z)) is close to 0 for an untrained G

loss_original = torch.log(1 - y)
grad_original = torch.autograd.grad(loss_original, y, retain_graph=True)[0]

y2 = torch.tensor(0.02, requires_grad=True)
loss_nonsat = torch.log(y2)
grad_nonsat = torch.autograd.grad(loss_nonsat, y2)[0]

print(f'd/dy log(1-y) at y=0.02:  {grad_original.item():.4f}  (original minimax loss)')
print(f'd/dy log(y)   at y=0.02:  {grad_nonsat.item():.4f}  (non-saturating loss)')
print(f'Ratio: the non-saturating gradient is {abs(grad_nonsat.item() / grad_original.item()):.1f}x larger in magnitude here.')
assert abs(grad_nonsat.item()) > abs(grad_original.item())
print('✓ Confirms Module 3: the non-saturating loss gives a far steeper gradient exactly where the original loss goes flat.')

---
# Part 3 — DCGAN Generator and Discriminator (`gan-m5`, `gan-m10`)

Following the DCGAN guidelines: strided/transposed convolutions instead of pooling, BatchNorm everywhere except the two pixel-facing layers, ReLU in G, LeakyReLU in D.

In [ ]:
LATENT_DIM = 100

class Generator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 256, kernel_size=7, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 1, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z.view(z.size(0), -1, 1, 1))


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 1, kernel_size=7, stride=1, padding=0, bias=False),
        )

    def forward(self, x):
        return self.net(x).view(-1)

G = Generator().to(DEVICE)
D = Discriminator().to(DEVICE)

z_test = torch.randn(4, LATENT_DIM, device=DEVICE)
fake_test = G(z_test)
print(f'G(z) output shape: {tuple(fake_test.shape)}  (expected (4, 1, 28, 28))')
assert tuple(fake_test.shape) == (4, 1, 28, 28)

d_test = D(fake_test)
print(f'D(x) output shape: {tuple(d_test.shape)}  (expected (4,))')
assert tuple(d_test.shape) == (4,)
print('✓ Generator and Discriminator shapes verified.')

---
# Part 4 — Training on Fashion-MNIST (`gan-m10`)

D is trained on real and fake batches separately; G is trained against the non-saturating loss from Module 3.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),  # scale to [-1, 1] to match Generator's Tanh output
])
train_dataset = datasets.FashionMNIST(root='./data', train=True, transform=transform, download=True)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, drop_last=True)

criterion = nn.BCEWithLogitsLoss()
opt_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

def train_step(real_images):
    batch_size = real_images.size(0)
    real_labels = torch.ones(batch_size, device=DEVICE)
    fake_labels = torch.zeros(batch_size, device=DEVICE)

    opt_D.zero_grad()
    d_real = D(real_images)
    loss_d_real = criterion(d_real, real_labels)

    z = torch.randn(batch_size, LATENT_DIM, device=DEVICE)
    fake_images = G(z)
    d_fake = D(fake_images.detach())
    loss_d_fake = criterion(d_fake, fake_labels)

    loss_d = loss_d_real + loss_d_fake
    loss_d.backward()
    opt_D.step()

    opt_G.zero_grad()
    d_on_fake = D(fake_images)
    loss_g = criterion(d_on_fake, real_labels)  # non-saturating trick (Module 3)
    loss_g.backward()
    opt_G.step()

    return loss_d.item(), loss_g.item()

EPOCHS = 5
print(f'Training DCGAN on Fashion-MNIST for {EPOCHS} epochs...')
for epoch in range(1, EPOCHS + 1):
    d_losses, g_losses = [], []
    for real_images, _ in train_loader:
        real_images = real_images.to(DEVICE)
        ld, lg = train_step(real_images)
        d_losses.append(ld)
        g_losses.append(lg)
    print(f'Epoch {epoch}/{EPOCHS} | D loss: {np.mean(d_losses):.3f} | G loss: {np.mean(g_losses):.3f}')

---
# Part 5 — Generated Samples After Training (`gan-m10`)

In [ ]:
@torch.no_grad()
def generate_samples(model, num_samples=16, device='cpu'):
    model.eval()
    z = torch.randn(num_samples, LATENT_DIM, device=device)
    samples = model(z)
    model.train()
    return (samples + 1) / 2  # rescale from [-1, 1] to [0, 1] for display

samples = generate_samples(G, num_samples=16, device=DEVICE)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i].cpu().squeeze(), cmap='gray')
    ax.axis('off')
plt.suptitle('DCGAN Samples Generated from Random z ~ N(0, I)', fontsize=13)
plt.tight_layout()
plt.show()

---
# Part 6 — WGAN-GP Gradient Penalty on a Toy Critic (`gan-m6`)

We verify the gradient-penalty mechanics directly: sample points between real and fake data, and penalize the critic's gradient norm for straying from 1.

In [ ]:
def gradient_penalty(critic, real, fake, device):
    batch_size = real.size(0)
    epsilon = torch.rand(batch_size, 1, 1, 1, device=device)
    interpolated = (epsilon * real + (1 - epsilon) * fake).requires_grad_(True)

    scores = critic(interpolated)
    grads = torch.autograd.grad(
        outputs=scores, inputs=interpolated,
        grad_outputs=torch.ones_like(scores),
        create_graph=True, retain_graph=True,
    )[0]

    grad_norm = grads.view(batch_size, -1).norm(2, dim=1)
    penalty = ((grad_norm - 1) ** 2).mean()
    return penalty, grad_norm

toy_critic = Discriminator().to(DEVICE)
real_batch = next(iter(train_loader))[0][:16].to(DEVICE)
with torch.no_grad():
    fake_batch = G(torch.randn(16, LATENT_DIM, device=DEVICE))

gp, grad_norm = gradient_penalty(toy_critic, real_batch, fake_batch, DEVICE)
print(f'Mean gradient norm on interpolated points (untrained critic): {grad_norm.mean().item():.3f}')
print(f'Gradient penalty term ((||grad|| - 1)^2, averaged):           {gp.item():.3f}')
print('An untrained critic will not sit at norm 1 yet — this term is exactly what WGAN-GP adds to the loss to push it there during training.')

---
# Part 7 — CycleGAN's Cycle-Consistency Loss (`gan-m8`)

Module 8's key claim: cycle-consistency loss alone is trivially satisfied by the identity mapping, which is exactly why the adversarial losses are still required. We verify this directly rather than training a second coupled GAN pair.

In [ ]:
def cycle_consistency_loss(F_fn, G_fn, x):
    # Round trip x -> G(x) -> F(G(x)), compared back to x. No paired ground truth involved.
    return (F_fn(G_fn(x)) - x).abs().mean()

identity = lambda t: t  # the trivial, do-nothing "generator"
x_sample = torch.randn(8, 3, 16, 16)

trivial_loss = cycle_consistency_loss(identity, identity, x_sample)
print(f'Cycle-consistency loss with G = F = identity: {trivial_loss.item():.8f}')
assert trivial_loss.item() < 1e-6, 'The identity mapping must trivially satisfy cycle-consistency'
print('Confirms Module 8: a perfect round trip is achieved by doing nothing at all.')

# A generator that scrambles content but is still perfectly invertible also satisfies the loss --
# a fixed permutation of pixels, undone exactly by its inverse permutation.
perm = torch.randperm(x_sample.numel() // x_sample.size(0))
inv_perm = torch.argsort(perm)

def permute_generator(t, order):
    flat = t.view(t.size(0), -1)
    return flat[:, order].view_as(t)

scramble = lambda t: permute_generator(t, perm)
unscramble = lambda t: permute_generator(t, inv_perm)

scrambled_loss = cycle_consistency_loss(unscramble, scramble, x_sample)
print(f'Cycle-consistency loss with an invertible pixel-scramble: {scrambled_loss.item():.8f}')
assert scrambled_loss.item() < 1e-5
print('An invertible but content-destroying mapping also satisfies cycle-consistency perfectly --')
print('this is exactly why the adversarial losses on D_X and D_Y, not cycle-consistency, are what')
print('force each generator output to actually resemble the target domain.')

---
# Part 8 — Toy Inception Score / FID Proxies (`gan-m9`)

Reproducing the module's point with numbers: a generator that is sharp but low-diversity keeps a respectable IS while its FID-style distance blows up.

In [ ]:
def toy_inception_score(sharpness, diversity):
    confidence = 0.3 + 0.7 * (sharpness / 100)
    spread = 0.6 + 0.4 * (diversity / 100)
    return round(float(np.exp(confidence) * spread), 2)

def toy_fid(sharpness, diversity):
    sharpness_gap = (100 - sharpness) / 100
    diversity_gap = ((100 - diversity) / 100) ** 2
    return round(float(sharpness_gap * 40 + diversity_gap * 220 + 3), 1)

healthy = dict(sharpness=80, diversity=85)
collapsed = dict(sharpness=90, diversity=20)  # sharp, but collapsed onto a few modes

for name, cfg in [('Healthy generator', healthy), ('Collapsed generator', collapsed)]:
    is_score = toy_inception_score(**cfg)
    fid_score = toy_fid(**cfg)
    print(f'{name:22s} | IS = {is_score:5.2f} (higher better) | FID = {fid_score:6.1f} (lower better)')

print('\nNote how much less IS moves than FID between the two rows — exactly the Module 8 point:')
print('IS barely penalises the collapse, FID reacts sharply because it compares whole distributions.')

---
# Part 9 — Latent-Space Interpolation on the Trained Generator (`gan-m11`)

In [ ]:
torch.manual_seed(7)
z_a = torch.randn(1, LATENT_DIM, device=DEVICE)
z_b = torch.randn(1, LATENT_DIM, device=DEVICE)

steps = 10
alphas = np.linspace(0, 1, steps)

fig, axes = plt.subplots(1, steps, figsize=(16, 2))
G.eval()
with torch.no_grad():
    for i, a in enumerate(alphas):
        z_interp = (1 - a) * z_a + a * z_b
        img = ((G(z_interp) + 1) / 2).cpu().squeeze().numpy()
        axes[i].imshow(img, cmap='gray')
        axes[i].axis('off')
        axes[i].set_title(f'{a:.1f}', fontsize=8)
G.train()
plt.suptitle('Latent Walk z_A → z_B Through the Trained Generator', fontsize=12, y=1.08)
plt.tight_layout()
plt.show()
print('A smooth walk with no abrupt jump confirms Module 11: G learned a continuous mapping over the whole latent space, not isolated points.')